In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

#### Hypothesis 1 


Songs with artist collaborations exhibit greater chart longevity than solo tracks, even after controlling for initial popularity and artist characteristics.

Analysis:
Run a multiple linear regression
- input: 
    - collaboration type (dummy variable: 0 = solo, 1 = featured collaboration)
    - number of featured artists
    - peak streams in Week 1 (as a control to isolate longevity from the initial spike)
- output: total weeks on chart

Test whether collaboration > 0, with a significance of 0.05. If collaboration  is significant after controlling for Week 1 streams, it confirms a sustained longevity effect rather than a mere spike.



In [56]:
df = pd.read_csv("new_data.csv")

In [57]:
# prep data for h1
# each row represents a song. each song should only appear once

df['date'] = pd.to_datetime(df['date'])

# find the stream of the song in the first week in top 50
first_appearance = (
    df.sort_values('date')
      .groupby(['uri', 'track_name'], as_index=False)
      .first()                          
)
first_appearance = first_appearance.rename(columns={'streams': 'streams_first_week',
                                                     'date':    'first_chart_date',
                                                     'rank':    'entry_rank_in_top50'})     

# count the weeks the song was in the top 50 chart
total_weeks_on_chart = df.groupby(["uri", "track_name"])["rank"].count().reset_index(name = 'total_weeks_on_chart')

df_h1 = pd.merge(total_weeks_on_chart, first_appearance, on = ["uri", "track_name"])

In [58]:
df_h1.columns

Index(['uri', 'track_name', 'total_weeks_on_chart', 'entry_rank_in_top50',
       'artist_names', 'source', 'peak_rank', 'previous_rank',
       'weeks_on_chart', 'streams_first_week', 'first_chart_date', 'tempo',
       'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_rolloff',
       'mfcc_1', 'mfcc_2', 'chroma_mean', 'chroma_std', 'artist_count',
       'collaboration_type', 'monthly_listeners', 'popularity_score'],
      dtype='str')

In [ ]:
x = df_h1[['streams_first_week', 'collaboration_type', 'entry_rank_in_top50', 'popularity_score']]
X = sm.add_constant(x)
Y = df_h1['total_weeks_on_chart']

model_h1 = sm.OLS(Y, X).fit()
results_h1 = model_h1.summary()
results_h1

# fail to reject, no evidence of collaboration effect on longevity 

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     total_weeks_on_chart   R-squared:                       0.053
Model:                              OLS   Adj. R-squared:                  0.050
Method:                   Least Squares   F-statistic:                     21.87
Date:                  Thu, 23 Apr 2026   Prob (F-statistic):           1.39e-17
Time:                          13:28:20   Log-Likelihood:                -6320.5
No. Observations:                  1580   AIC:                         1.265e+04
Df Residuals:                      1575   BIC:                         1.268e+04
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  19.3319      3.167      6.104      0.000      13.120      25.544
streams_first_week   1.719e-07   3.99e-08      4.306      0.000    9.36e-08     2.5e-07
collaboration_type     -0.8491      0.693     -1.225      0.221      -2.209       0.510
entry_rank_in_top50    -0.0902      0.032     -2.841      0.005      -0.152      -0.028
popularity_score       -0.1383      0.032     -4.276      0.000      -0.202      -0.075
==============================================================================
Omnibus:                     1409.329   Durbin-Watson:                   2.049
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            50663.372
Skew:                           4.114   Prob(JB):                         0.00
Kurtosis:                      29.493   Cond. No.                     2.29e+08
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.29e+08. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

#### Hypothesis 2: 

Songs released under major labels have greater chart longevity than independent releases, even after controlling for both initial popularity and musical characteristics.

Analysis: 
Run a Multiple Linear Regression
- Output (Dependent Variable): Total weeks on chart
- Input Variables:
    - label_type : dummy variable (0 = independent, 1 = major label), which is the key variable of interest
    - week1_streams :  peak streams in Week, controls for initial popularity
    - danceability, energy, tempo, valence, acousticness :  control for musical characteristics that independently drive longevity

Test: Whether the coefficient for label_type is significantly positive at α = 0.05. If significant after controlling for Week 1 streams and musical features, it confirms a structural label advantage (e.g., marketing budget, playlist placement) rather than just better music or an initial spike.





#### Hypothesis 3: 

The likelihood of a song becoming a "viral hit" (reaching the Top 15) is significantly higher when specific levels of energy and spectral brightness are combined, rather than the independent effect of either feature alone. 

Analysis: 
Run a Logistic Regression to predict the probability of a song reaching the Top 15

- Input: energy (numerical), spectral_centroid (numerical), and an interaction term (energy × spectral_centroid).
- Output: Viral Status (Dummy variable: 1 if peak_rank is 1–15, 0 if 16–50)

Test: We will test whether the coefficient for the interaction term (interaction) is significantly different from zero ( < 0.05). If significant, it confirms that a certain combo (e.g., high energy plus high brightness) creates a unique synergy that drives virality more effectively than just having one or the other.


In [ ]:
# prep data for h3
# each row represents a song, each song should only appear once

df['viral'] = np.where(df['peak_rank'] <= 15, 1, 0)
df_h3 = df.drop_duplicates(subset = ['uri', 'track_name'])

# interaction term 
# add controls ? 

In [ ]:
x = df_h3[['energy', 'spectral_centroid', 'energy_spectral_centroid']]
Y = df_h3['viral']
lr_h3 = LogisticRegression().fit(x, Y)